In [4]:
#Import packages
import sys 
sys.path.append('./New functions & Dash GUI')
from dash import Dash, html, dash_table, dcc, callback, Output, Input, State, no_update, ctx
import dash
import os
from dash import ctx
import dash_uploader as du
import numpy as np
import astropy.units as u
import pandas as pd
import plotly.express as px
import csv
import base64
import io
import datetime
import glob
import warnings
from astropy.io.fits.verify import VerifyWarning
warnings.simplefilter('ignore', category=VerifyWarning)
import traceback
from astropy.io import fits
from dash import Output, Input, callback, no_update 
from IPython.display import FileLink
import plotly.graph_objects as go
import importlib
from itertools import zip_longest
from astropy.io import fits
from astropy.table import Table, QTable, hstack
from matplotlib.colors import LogNorm
import dash_bootstrap_components as dbc
from dash_bootstrap_templates import load_figure_template
import galaxy_functions_Copy2

importlib.reload(galaxy_functions_Copy2)

from galaxy_functions_Copy2 import ElementInput, redshifted_wavelengthFunc, line_flux_at_z, sensScaled, Detected_Galaxies
df_rest_wave = pd.read_excel('/data/projects/FIRESIDE/data/Rest_Wave_data.xlsx', header=0)
rest_wave_table = Table.from_pandas(df_rest_wave)


#Load in data catalog
#line_table_data = fits.open('/data/projects/FIRESIDE/data/modular_pySIDES_plus_AGN_Q_alpha_0.5(copy).fits')

In [5]:
#Initialize the Dash App
app = Dash(__name__, external_stylesheets=[dbc.themes.SLATE])
app.config.suppress_callback_exceptions = True
load_figure_template("minty")
#app.validation_layout = html.Div()

#Folder for storing data catalogs on pc & Folder for downloaded FITS from Dash 
CATALOG_ROOT = "/data/projects/FIRESIDE/data/data_catalogs"
DOWNLOAD_FOLDER = "/data/projects/FIRESIDE/data/Dash_FITS_downloads"
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

#Folder for storing configuartion sheets on pc
CONFIGS_ROOT = "/data/projects/FIRESIDE/data/FIRESIDE_Configs_sheets"
os.makedirs(CONFIGS_ROOT, exist_ok=True) 

def fits_file_dropdown():
    #Check data_catalogs folder for FITS files
    file_type = os.path.join(CATALOG_ROOT, "*.fit*")
    file_path = glob.glob(file_type)
    
    file_path_id = []
    for fits in file_path:
        file_name = os.path.basename(fits)
        
        file_path_id.append({"label": file_name, "value": fits})
        
    print("JUPYTER CURRENT DIRECTORY:")
    print(os.getcwd())

    print("\nWHERE GLOB IS ACTUALLY LOOKING FOR FILES:")
    print(CATALOG_ROOT) 
        
    return file_path_id

def configuartion_sheets():
        
    file_type = os.path.join(CONFIGS_ROOT, "*.ecsv*")
    file_path = glob.glob(file_type)
    
    file_path_id = []
    for configs in file_path:
        file_name = os.path.basename(configs)
        label_name, file_format = os.path.splitext(file_name)
        
        file_path_id.append({"label": label_name, "value": configs})
    return file_path_id


#Tab 1 components 
tab1_layout = dbc.Container(
    fluid=True,
    children=[dbc.Row(className="mb-4", justify ='end',
            children=[dbc.Col(
                    width=9,
                    children=[dbc.Card(
                            body=True,
                            color="secondary",
                            outline=True,
                            children=[
                                dbc.Row([dbc.Col([html.Div('Instrument Specs', className='text-info fw-bold fs-5', style={'textShadow':'0 0 1px #fff'})], width=4), 
                                dbc.Col(width=4), 
                                dbc.Col([html.Div('Survey Parameters', className='text-info fw-bold fs-5' , style={'textShadow':'0 0 1px #fff'})], width=4)]), 
                                html.Hr(className='my-3'), 
                                #Wavelength & Area Constraints
                                dbc.Row([dbc.Col([html.Label('Min Wavelength (μm)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='waveMin', type='number', value=None, placeholder='Enter min λ', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                    
                                dbc.Col([html.Label('Sensitivity (W/m^2)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='sensitivity', type='number', value=None, placeholder='Enter Inst. Sensitivity', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4, className="border-end border-light border-opacity-25"),
                                dbc.Col([html.Label('Observed Area (sq. deg)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='area', type='number', value=1, placeholder='Enter Size Obsv. Area', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                            
                                dbc.Col([html.Label('Max Wavelength (μm)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='waveMax', type='number', value=None, placeholder='Enter max λ', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                dbc.Col([html.Label('FOV (sq. deg)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='fov', type='number', value=None, placeholder='Enter FOV', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4, className="border-end border-light border-opacity-25"),
                                dbc.Col([html.Label('Observation Time (hr)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='tobsv', type='number', value=1000, placeholder='Enter Obsv. Time', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4)])])])]),




              
        dbc.Row(children=[dbc.Col(width=3, style={'marginTop':'3px'},
                    children=[dbc.Card(body=True,
                            color="secondary",
                            outline=True,
                            children=[
                                    
                                #Emission Line Selection
                                html.Label('Instrument Band(s)', className="fw-bold text-white mt-2"), 

                                html.Div([
        html.Button('Clear ⟲', id='clear_inputs', n_clicks=0, 
        className='btn btn-light btn-sm rounded-0 ms-auto',
            style={
                "padding": "1px 6px",
                "fontSize": "10px", 
                "color": "#000000",
                "fontWeight": "500",
                "borderBottom": "none",
                "letterSpacing": "0.3px"}), ],
                                        
    className="d-flex align-items-end w-100 mt-2" #leave in, aligns actual box for upper html. Will break otherwise!
),

                                
                                dcc.Dropdown(options=[], value=None, placeholder='Select Band', id='band_selection', multi=True, className="text-dark mb-3"),

                                html.Label('Viewing Parameters:', className="fw-bold text-white d-inline-block mt-2"), 
                                html.Span("ⓘ", id="viewing_parameters_info", className="text-info fw-bold ms-2", style={"cursor": "help", "fontSize": "14px"}),
                                dbc.Tooltip(html.Span([html.B("Visual Only - "), "Use to update Instrument Specs"]),
                                target="viewing_parameters_info", placement="top-end", className="p-2"),
                                
                                
                                
                    dbc.RadioItems(id='active_band_toggle', options=[], value=None, inline=True, 
                    className='p-1 bg-transparent text-white', inputClassName="form-check-input bg-transparent border-secondary",
    inputStyle={
        "accent-color": "#39FF14",
        "background-color": "initial",      
        "border-color": "initial"},
    labelStyle={
        'display': 'inline-block', 
        'margin-right': '15px', 
        'cursor': 'pointer',
        'font-weight': '500', 
        'color': '#ffffff'
    }),
                                
                                html.Label('Select Line Emission', className="fw-bold text-white mt-2"),
                                dcc.Dropdown(options=[], value=None, placeholder='Select Emission Line', id='controls-and-radio-item', multi=False, className="text-dark mb-3"),

                                dbc.Accordion([dbc.AccordionItem(title=html.Span("AGN Fraction", className="text-white fw-bold"), 
                                                                 
                                children=[
                                html.Label('Upper AGN fraction', className="fw-bold text-white"),
                                dcc.Input(id='upper_f_agn_1', type='number', value=1.00, placeholder='Enter upper AGN fraction limit', className="form-control",
                                style={"padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"}),

                                html.Label('Lower AGN fraction', className="fw-bold text-white"),
                                dcc.Input(id='lower_f_agn_1', type='number', value=0.00, placeholder='Enter lower AGN fraction limit', className="form-control", 
                                style={"padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})])]), 
                                
                                html.Hr(),
                                
                                #Graph Settings
                    dbc.Accordion([dbc.AccordionItem(title=html.Span("Graphing", className="text-white fw-bold"),
                    children=[
                    html.Div([
                                html.Label('X-axis', className="fw-bold text-white"),
                                dcc.Dropdown(options=[], value=None, placeholder='Select X-axis', id='dropdown_x', className="text-dark mb-2"), 
                                
                                html.Label('Y-axis', className="fw-bold text-white"),
                                dcc.Dropdown(options=[], value=None, placeholder='Select Y-axis', id='dropdown_y', className="text-dark mb-2"),
                                
                                html.Label('Graph-type', className="fw-bold text-white"),
                                dcc.Dropdown(['Histogram', 'XY-Graph', '2D-Histogram'], 'Histogram', placeholder='Select Graph', id='graph_type', className="text-dark mb-2"), 
                        ], className="border border-secondary rounded p-3 mb-3")])], start_collapsed=True), 

                  
                    html.Hr(), 

                                dbc.Accordion([dbc.AccordionItem(title=html.Span("Scaling & Tools", className="text-white fw-bold"),
                                children=[
                                html.Div([
                                html.Label('Axis Scale', className="fw-bold text-white"),
                                dbc.RadioItems(options=['All Linear', 'All Log', 'X-Log', 'Y-Log'], value='All Linear', id='axis_type',className='p-1 text-white', 
                                inputClassName="form-check-input bg-transparent border-secondary",
               
    style={
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "columnGap": "20px",
        "rowGap": "12px"},
    labelStyle={
        "white-space": "nowrap",  
        "display": "inline-flex",
        "align-items": "center",
        "gap": "8px"}
), ], className="border border-secondary rounded p-3 mb-3"),  

                                html.Div([         
                                html.Label('Coloring Options', className="fw-bold text-white"),
                                dcc.RadioItems(options=['Band', 'Line'], value='Band', id='coloring', inline=True, 
                                className='p-1 bg-transparent text-white', 
                                inputClassName="form-check-input bg-transparent border-secondary",  inputStyle={
        "accent-color": "#39FF14",
        "background-color": "initial",      
        "border-color": "initial"},
    labelStyle={
        'display': 'inline-block', 
        'margin-right': '15px', 
        'cursor': 'pointer',
        'font-weight': '500', 
        'color': '#ffffff'
})], className="border border-secondary rounded p-3 mb-3"),

html.Div([
    html.Div([
        html.Span("ⓘ", id="Hist_Tools_info", className="text-info fw-bold ms-2", style={"cursor": "help", "fontSize": "14px"}),
        dbc.Tooltip(html.Div([
            html.B("Survey Curve - "), "Show/hide the binned distribution profile line of filtered detections", 
            html.Br(),
            html.B("All Sources - "), "Show/hide the binned distribution profile line of detections for all galaxies"
        ]), 
        target="Hist_Tools_info", placement="top-end", className="p-2")], 
        style={"position": "absolute", "top": "10px", "right": "15px"}),

    html.Label('Histogram Tools', className="fw-bold text-white"),
    dbc.Checklist(
        options=['Survey Curve', 'All Sources'], 
        value=[], id='Hist_Tools', inline=True, 
        className='p-1 bg-transparent text-white', inputClassName="form-check-input bg-transparent border-secondary",  
        inputStyle={
            "accent-color": "#39FF14",
            "background-color": "initial",      
            "border-color": "initial"
        },
        labelStyle={
            'display': 'inline-block', 
            'margin-right': '15px', 
            'cursor': 'pointer',
            'font-weight': '500', 
            'color': '#ffffff'})],
        className="border border-secondary rounded p-3", style={"position": "relative"}) 
])])]),
                                
                                #Buttons
                                html.Hr(),
                                dbc.Button("Plot Graph", id="graph_button", n_clicks=0, color="info", className="w-100 mb-2"),
                                dbc.Button("FITS Download Detections", id="FITS_download", n_clicks=0, color="success", className="w-100"), 
                                dcc.Download(id='Filtered_galaxies')]),
                
                dbc.Col(width=9,
                    children=[dbc.Card(body=True,
                            children=[
                                dcc.Graph(figure={}, id='controls-and-graph', style={"height": "650px"})])])])])


#Tab 2 MULTI components 
tab2_layout = dbc.Container(fluid=True,
    children=[dbc.Row(className="mb-4", justify ='end',
            children=[dbc.Col(
                    width=9,
                    children=[dbc.Card(
                            body=True,
                            color="secondary",
                            outline=True,
                            children=[dbc.Row([dbc.Col([html.Div('Instrument Specs', className='text-info fw-bold fs-5', style={'textShadow':'0 0 1px #fff'})], width=4), 
                                dbc.Col(width=4), 
                                dbc.Col([html.Div('Survey Parameters', className='text-info fw-bold fs-5', style={'textShadow':'0 0 1px #fff'})], width=4)]), 
                                html.Hr(className='my-3'), 
                                #Wavelength & Area Constraints
                                dbc.Row([dbc.Col([html.Label('Min Wavelength (μm)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='waveMin', type='number', value=None, placeholder='Enter min λ', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                    
                                dbc.Col([html.Label('Sensitivity (W/m^2)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='sensitivity', type='number', value=None, placeholder='Enter Inst. Sensitivity', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4, className="border-end border-light border-opacity-25"),
                                dbc.Col([html.Label('Observed Area (sq. deg)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='area', type='number', value=1, placeholder='Enter Size Obsv. Area', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                            
                                dbc.Col([html.Label('Max Wavelength (μm)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='waveMax', type='number', value=None, placeholder='Enter max λ', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4),
                                dbc.Col([html.Label('FOV (sq. deg)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='fov', type='number', value=None, placeholder='Enter FOV', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4, className="border-end border-light border-opacity-25"),
                                dbc.Col([html.Label('Observation Time (hr)', className="fw-bold text-white fs-6 text-nowrap"), dcc.Input(id='tobsv', type='number', value=1000, placeholder='Enter Obsv. Time', className="form-control", style={"width": "80%", "padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})], width=4)])])])]),
              

        dbc.Row(children=[dbc.Col(width=3, style={'marginTop':'3px'},
                    children=[dbc.Card(
                            body=True,
                            color="secondary",
                            outline=True,
                            children=[

                                html.Label('Instrument Band(s)', className="fw-bold text-white mt-2"), 

                                html.Div([
                                html.Button('Clear ⟲', id='clear_inputs_2', n_clicks=0, 
                                className='btn btn-light btn-sm rounded-0 ms-auto',
            style={
                "padding": "1px 6px",
                "fontSize": "10px", 
                "color": "#000000",
                "fontWeight": "500",
                "borderBottom": "none",
                "letterSpacing": "0.3px"}), ],
                                        
    className="d-flex align-items-end w-100 mt-2" #leave in, aligns actual box for upper html. Will break otherwise!
),

                                
                                dcc.Dropdown(options=[], value=None, placeholder='Select Band', id='band_selection_2', multi=True, className="text-dark mb-3"),
                                
                                html.Label('Viewing Parameters:', className="fw-bold text-white d-inline-block mt-2"), 
                                html.Span("ⓘ", id="viewing_parameters_info", className="text-info fw-bold ms-2", style={"cursor": "help", "fontSize": "14px"}),
                                dbc.Tooltip(html.Span([html.B("Visual Only - "), "Use to update Instrument Specs"]),
                                target="viewing_parameters_info", placement="top-end", className="p-2"),
                                
    dbc.RadioItems(id='active_band_toggle_2', options=[], value=None, inline=True, className='p-1 bg-transparent text-white', inputClassName="form-check-input bg-transparent border-secondary",
    inputStyle={
        "accent-color": "#39FF14",
        "background-color": "initial",      
        "border-color": "initial"},
    labelStyle={
        'display': 'inline-block', 
        'margin-right': '15px', 
        'cursor': 'pointer',
        'font-weight': '500', 
        'color': '#ffffff'
    }),
                                
                                #Element Selection 
                                html.Label('Select Line Emission(s)', className="fw-bold text-white mt-2"),
                                #dropdown
                                dcc.Dropdown(options=[], value=None, placeholder='Select Emission Line', id='multi-controls-and-radio-item', multi=True, className="text-dark mb-3"),

                                dbc.Accordion([dbc.AccordionItem(title=html.Span("AGN Fraction", className="text-white fw-bold"), 
                                
                                children=[
                                html.Label('Upper AGN fraction', className="fw-bold text-white"),
                                dcc.Input(id='upper_f_agn', type='number', value=1.00, placeholder='Enter upper AGN fraction limit', className="form-control",
                                style={"padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"}),

                                html.Label('Lower AGN fraction', className="fw-bold text-white"),
                                dcc.Input(id='lower_f_agn', type='number', value=0.00, placeholder='Enter lower AGN fraction limit', className="form-control", 
                                style={"padding": "0.375rem 0.75rem", "lineHeight": "1.5", "textAlign": "center"})])]),
                                
                                html.Hr(),
                                
        #Graph Settings
dbc.Accordion([dbc.AccordionItem(title=html.Span("Graphing", className="text-white fw-bold"),
                                                                 
                    children=[
                    html.Div([
                    html.Label("X-axis", className="fw-bold text-white"),
                    dcc.Dropdown(options=[], value=None, placeholder="Select X-axis", id="multi_dropdown_x", className="text-dark mb-2"),
                    html.Label("Y-axis", className="fw-bold text-white"),
                    dcc.Dropdown(options=[], value=None, placeholder="Select Y-axis", id="multi_dropdown_y", className="text-dark mb-2"),
                    html.Label("Graph-type", className="fw-bold text-white"),
                    dcc.Dropdown(["Histogram", "XY-Graph", "2D-Histogram"], "Histogram", placeholder="Select Graph", id="graph_type", className="text-dark mb-2")
                ],  className="border border-secondary rounded p-3 mb-3")])]),
                                html.Hr(), 
            
 
                                    
                                dbc.Accordion([dbc.AccordionItem(title=html.Span("Scaling & Tools", className="text-white fw-bold"),
                                                                 
                                children=[
                                html.Div([
                                html.Label('Axis Scale', className="fw-bold text-white"),
dbc.RadioItems(options=['All Linear', 'All Log', 'X-Log', 'Y-Log'], value='All Linear', id='axis_type',className='p-1 text-white', 
    inputClassName="form-check-input bg-transparent border-secondary",
               
    style={
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "columnGap": "20px",
        "rowGap": "12px"},
    labelStyle={
        "white-space": "nowrap",  
        "display": "inline-flex",
        "align-items": "center",
        "gap": "8px"}
), ], className="border border-secondary rounded p-3 mb-3"),
                                    
                                html.Div([
                                html.Label('Coloring Options', className="fw-bold text-white"),
                                dcc.RadioItems(options=['Line', 'Band', 'None'], value='Line', id='coloring_2', inline=True, 
                                className="p-1 bg-transparent text-white", inputClassName="form-check-input bg-transparent border-secondary",  inputStyle={
        "accent-color": "#39FF14",
        "background-color": "initial",      
        "border-color": "initial"},
    labelStyle={
        'display': 'inline-block', 
        'margin-right': '15px', 
        'cursor': 'pointer',
        'font-weight': '500', 
        'color': '#ffffff'
     })], className="border border-secondary rounded p-3 mb-3"),

html.Div([
    html.Div([
        html.Span("ⓘ", id="Hist_Tools_info_2", className="text-info fw-bold ms-2", style={"cursor": "help", "fontSize": "14px"}),
        dbc.Tooltip(html.Div([
            html.B("Survey Curve - "), "Show/hide the binned distribution profile line of filtered detections", 
            html.Br(),
            html.B("All Sources - "), "Show/hide the binned distribution profile line of detections for all galaxies"
        ]), 
        target="Hist_Tools_info_2", placement="top-end", className="p-2")], 
        style={"position": "absolute", "top": "10px", "right": "15px"}),

    html.Label('Histogram Tools', className="fw-bold text-white"),
    dbc.Checklist(
        options=['Survey Curve', 'All Sources'], 
        value=[], id='Hist_Tools_2', inline=True, 
        className='p-1 bg-transparent text-white', inputClassName="form-check-input bg-transparent border-secondary",  
        inputStyle={
            "accent-color": "#39FF14",
            "background-color": "initial",      
            "border-color": "initial"
        },
        labelStyle={
            'display': 'inline-block', 
            'margin-right': '15px', 
            'cursor': 'pointer',
            'font-weight': '500', 
            'color': '#ffffff'})],
        className="border border-secondary rounded p-3 mb-3", style={"position": "relative"}), 

        html.Div([
            html.Div([
            html.Span("ⓘ", id="detections_info", className="text-info fw-bold ms-2", style={"cursor": "help", "fontSize": "14px"}),
            dbc.Tooltip(html.Div([
                html.B("Joint - "), "Filters detections by galaxies containing ALL common selected emission lines.", 
                html.Br(),
                html.B("Separate- "), "Filters detections by galaxies containing AT LEAST ONE selected emission line."
        ]), 
        target="detections_info", placement="top-end", className="p-2")], 
        style={"position": "absolute", "top": "10px", "right": "15px"}),
            
                html.Label('Detections Filter', className="fw-bold text-white"),
                dcc.RadioItems(options=['Joint', 'Separate'], value='Joint', id='detections_filter_2', inline=True, 
                className="p-1 bg-transparent text-white", inputClassName="form-check-input bg-transparent border-secondary",  inputStyle={
        "accent-color": "#39FF14",
        "background-color": "initial",      
        "border-color": "initial"},
    labelStyle={
        'display': 'inline-block', 
        'margin-right': '15px', 
        'cursor': 'pointer',
        'font-weight': '500', 
        'color': '#ffffff'
     })], className="border border-secondary rounded p-3 mb-3", style={"position": "relative"})
                                
                                ])])]),
                                
                                # Actions Section
                                html.Hr(),
                                dbc.Button("Plot Graph", id="graph_button_2", n_clicks=0, color="info", className="w-100 mb-2"),
                                dbc.Button("FITS Download Detections", id="FITS_download_2", color="success", className="w-100"), 
                                dcc.Download(id='Filtered_galaxies_2')]),
                
                dbc.Col(width=9,
                    children=[dbc.Card(
                            body=True,
                            children=[
                                dcc.Graph(figure={}, id='controls-and-graph_2', style={"height": "650px"})])])])])



app.layout = html.Div([
    dbc.Container(
        fluid=True,
        className="p-0",
        children=[dbc.Row(
                className="mb-0 p-3 bg-dark rounded", style={"marginTop":"0px"},
                children=[
                    dbc.Col(html.H1('FIRESIDE PHASE I', className="text-black m-0 fw-bold", style={"fontSize":"50px","lineHeight":"2.1", '-webkitTextStroke':'2px #ffffff'}),width=8),
                    # Pictures
                    dbc.Col(width=4,
                        style={"display": "flex", "justifyContent": "flex-end", "alignItems": "center"},
                        children=[
                            html.Img(
                                src="/assets/Galaxy.png",
                                style={"height": "100px",
                                    "borderRadius": "50%",
                                    "animation": "spin 8s linear infinite", "marginRight":"230px"}), 
                            
                        html.Img(src="/assets/fireside_light.png",
                                style={"height": "60px", "marginRight": "20px", "borderRadius": "4px"}),
                        
                        html.Img(
                                src="/assets/NASA-Logo-Large.png",
                                style={"height": "60px", "marginRight": "15px", "borderRadius": "4px"}), 
                        html.Img(
                                src="/assets/ipac.png",
                                style={"height": "50px", "marginRight": "15px", "borderRadius": "4px"}), 
                        html.Img(
                                src="/assets/Tufts_University_logo.png",
                                style={"height": "50px", "marginRight": "15px", "borderRadius": "4px"}),
                        html.Img(
                                src="/assets/stsci.png",
                                style={"height": "60px", "marginRight": "15px", "borderRadius": "4px"}),
                        html.Img(src="/assets/Winona.png",
                                style={"height": "60px", "marginRight": "15px", "borderRadius": "4px"}),
                    
])])]),
    
    #Folder selection for FITS catalogs
dbc.Card(
    body=True,
    color="secondary",
    outline=True,
    className="my-3 mx-auto shadow-sm border border-secondary",
    style={
        "width": "23.4%", "border": "1px solid", "borderRadius": "10px", "position": "relative", "top": "26px", "left": "-37.5%", "zIndex": "10"},
    children=[
        html.Div(
            className="d-flex justify-content-center align-items-center mb-2",
            children=[
                html.Span(
                    "Catalog Selection",
                    className="fw-bold text-white ms-3",
                    style={"fontSize": "13px", "opacity": "1"}),
                html.Span(
                    "ⓘ",
                    id="catalog_dropdown_info",
                    className="text-info fw-bold ms-auto me-2",
                    style={"cursor": "help", "fontSize": "14px"}),
                dbc.Tooltip(
                    "Select pre-loaded FITS from folder",
                    target="catalog_dropdown_info",
                    placement="top-end",
                    className="p-2")]),
        html.Div(
            style={
                "backgroundColor": "#2b303b",
                "borderRadius": "10px",
                "padding": "5px",
                "border": "1px solid", "marginBottom": "9px"},
            children=[
                dcc.Dropdown(
                    id="catalog_folder_selection",
                    options=fits_file_dropdown(),
                    value=fits_file_dropdown()[0]["value"] if fits_file_dropdown() else None,
                    placeholder="Select FITS from folder",
                    clearable=True,
                    style={
                        "backgroundColor": "#1e222b",
                        "color": "#000000",
                        "fontSize": "12px",
                        "borderColor": "#4f5b66",
                        "cursor": "pointer", "opacity": "1"})]),
        html.Hr(className='my-3'),
        html.Div(
            className="d-flex justify-content-center align-items-center mb-2",
            children=[
                html.Span(
                    "Configuration Presets",
                    className="fw-bold text-white ms-3",
                    style={"fontSize": "13px"}),
                html.Span(
                    "ⓘ",
                    id="configs_dropdown_info",
                    className="text-info fw-bold ms-auto me-2",
                    style={"cursor": "help", "fontSize": "14px"}),
                dbc.Tooltip(
                    "Select pre-loaded configurations from folder",
                    target="configs_dropdown_info",
                    placement="top-end",
                    className="p-2")]),
        html.Div(
            style={
                "backgroundColor": "#2b303b",
                "borderRadius": "10px",
                "padding": "5px",
                "border": "1px solid"},
            children=[
                dcc.Dropdown(
                    id="configs_selection",
                    options=configuartion_sheets(),
                    placeholder="Select configuration from folder",
                    value=configuartion_sheets()[0]["value"] if configuartion_sheets() else None,
                    clearable=True,
                    style={
                        "backgroundColor": "#1e222b",
                        "color": "#000000",
                        "fontSize": "12px",
                        "borderColor": "#4f5b66",
                        "cursor": "pointer"})])]),

    
    html.Div(style={"marginTop":"-250px"},
             children=[dbc.Tabs(id='graph_tabs', 
        active_tab='tab_single_element',
        className="nav-pills mt-0",
        style={'marginLeft':'25.80%'},
        children=[
            dbc.Tab(
                label='Single-Observable', 
                tab_id='tab_single_element',
                active_label_class_name="text-white bg-transparent border-0 fw-bold",
                label_class_name="bg-white text-secondary mx-1"),
            dbc.Tab(
                label='Multi-Observable', 
                tab_id='tab_combined_multi_element',
                active_label_class_name="text-white bg-transparent border-0 fw-bold",
                label_class_name="bg-white text-secondary mx-1")])]), 
    
    html.Div(id='tab_content'), 
    dcc.Store(id='function_data'), 
    dcc.Store(id='GalaxyCat_data'),
    dcc.Store(id='joint_store'), 
    dcc.Store(id='GalaxyCat_data_2'),
    dcc.Store(id='configs_data'),
    dcc.Store(id='band_data'),
    dcc.Store(id='catalog_data_store', data=None)])

JUPYTER CURRENT DIRECTORY:
/home/wag_user2

WHERE GLOB IS ACTUALLY LOOKING FOR FILES:
/data/projects/FIRESIDE/data/data_catalogs
JUPYTER CURRENT DIRECTORY:
/home/wag_user2

WHERE GLOB IS ACTUALLY LOOKING FOR FILES:
/data/projects/FIRESIDE/data/data_catalogs
JUPYTER CURRENT DIRECTORY:
/home/wag_user2

WHERE GLOB IS ACTUALLY LOOKING FOR FILES:
/data/projects/FIRESIDE/data/data_catalogs


In [6]:
#Tab callbacks 
@callback(
    Output(component_id='tab_content', component_property='children'),
    Input(component_id='graph_tabs', component_property='active_tab'))
def tab_identity(tab):
    if tab == 'tab_single_element':
        return tab1_layout
        
    elif tab == 'tab_combined_multi_element':
        return tab2_layout

    else:
        return html.Div('Error!')



    

In [7]:
@callback(
    Output(component_id='catalog_data_store', component_property='data'),
    Input(component_id='catalog_folder_selection', component_property='value'), 
    prevent_initial_call=False)

def dcc_data_catalog_store(selected_file_path):
    if not selected_file_path:
        return no_update
        
    file_store = selected_file_path
    
    return file_store

In [8]:
@callback(
    Output(component_id='controls-and-radio-item', component_property='options'), 
    Output(component_id='controls-and-radio-item', component_property='value'), 
    Input(component_id='catalog_data_store', component_property='data')) 
def update_single_dropdown(uploaded_catalog):
    
    #If no file, return nothing
    if not uploaded_catalog: 
        return [], None
        
        
    try: 
        
        file_path = uploaded_catalog
        
        #memmap=True reads headers instantly without consuming memory
        with fits.open(file_path, memmap=True) as hdul: 
            line_column_headers = []
            for line_headers in hdul:
                if hasattr(line_headers, 'columns') and line_headers.columns is not None:
                    line_column_headers = line_headers.columns.names
                    break
                    
        emission_lines = []
        for header in line_column_headers:
            if '[' in header:
                emission_lines.append(header)
        
        if len(emission_lines) == 0: 
            print("Error: Column headers must include '[Line Emission]'") 
            return [], None
            
        #Feed to dropdown
        return emission_lines, emission_lines[0]
        
    except Exception as error_headers: 
        print('Error reading FITS headers') 
        return ['Error reading file headers'], None


In [9]:
@callback(
    Output(component_id='multi-controls-and-radio-item', component_property='options'), 
    Output(component_id='multi-controls-and-radio-item', component_property= 'value'), 
    Input(component_id='catalog_data_store', component_property='data'),
    Input(component_id='graph_tabs', component_property='active_tab'))

 
def update_multi_dropdown(uploaded_catalog, active_tab): 
    
    #return empty lists when upload fails
    if not uploaded_catalog: 
        return [], None
    
    if active_tab == 'tab_single_element':
        return [], None
    try: 
        file_path = uploaded_catalog
        
        #Read headers instantly using memmap
        with fits.open(file_path, memmap=True) as hdul: 
            line_column_headers = []
            for line_headers in hdul:
                if hasattr(line_headers, 'columns') and line_headers.columns is not None:
                    line_column_headers = line_headers.columns.names
                    break  
            
        #Take headers from fits
        emission_lines = []
        for header in line_column_headers:
            if '[' in header:
                emission_lines.append(header)
        
        #return empty list when no headers found
        if len(emission_lines) == 0: 
            print("No matching Emission Lines found.") 
            return [], []
            
        return emission_lines, [emission_lines[0]]
        
    except Exception as error_headers: 
        traceback.print_exc()
        print(f"Error reading multi FITS headers: {error_headers}") 
        return ['Error reading file headers'], []


# Update Function 

In [10]:
#Callback for passing dash updates to galaxy function in underlying py
@callback(
    Output(component_id='function_data', component_property='data', allow_duplicate=True),
    Input(component_id='controls-and-radio-item', component_property='value', allow_optional=True),
    Input(component_id='multi-controls-and-radio-item', component_property='value', allow_optional=True),
    Input(component_id='graph_tabs', component_property='active_tab'),
    Input(component_id='catalog_data_store', component_property='data'),
    Input(component_id='band_data', component_property='data'),
    Input(component_id='waveMin', component_property='value', allow_optional=True),
    Input(component_id='waveMax', component_property='value', allow_optional=True), 
    Input(component_id='sensitivity', component_property='value', allow_optional=True),
    Input(component_id='tobsv', component_property='value', allow_optional=True),
    Input(component_id='fov', component_property='value', allow_optional=True),
    Input(component_id='area', component_property='value', allow_optional=True),
    Input(component_id='upper_f_agn_1', component_property='value', allow_optional=True),
    Input(component_id='lower_f_agn_1', component_property='value', allow_optional=True),
    Input(component_id='upper_f_agn', component_property='value', allow_optional=True),
    Input(component_id='lower_f_agn', component_property='value', allow_optional=True),
    Input(component_id='joint_store', component_property='data', allow_optional=True),
    prevent_initial_call=True)

def update_function(Emission_line_single, Emission_line_multi, active_tab, file_path, band_data, waveMin, waveMax, sensitivity, tobsv, fov, area, upper_f_agn_1, lower_f_agn_1, upper_f_agn, lower_f_agn, joint):
    if not file_path: 
         return dash.no_update

    if waveMin is not None:
        waveMin = float(waveMin)
    if waveMax is not None:
        waveMax = float(waveMax)
    if fov is not None:
        fov = float(fov)
    if sensitivity is not None:
        sensitivity = float(sensitivity)
    
    if active_tab == 'tab_single_element':
        up_f_agn = upper_f_agn_1
        low_f_agn = lower_f_agn_1
    else:
        up_f_agn = upper_f_agn
        low_f_agn = lower_f_agn

    #upper AGN fraction bounds
    if up_f_agn is not None:
        up_f_agn = float(up_f_agn)
        if up_f_agn < 0:
            up_f_agn = 0.0
        elif up_f_agn > 1:
            up_f_agn = 1.0

    #lower AGN fraction bounds
    if low_f_agn is not None:
        low_f_agn = float(low_f_agn)
        if low_f_agn < 0:
            low_f_agn = 0.0
        elif low_f_agn > 1:
            low_f_agn = 1.0

        
    try:
        
        if waveMin is None or waveMax is None or sensitivity is None or tobsv is None or area is None or fov is None or up_f_agn is None or low_f_agn is None:
                return dash.no_update
            
        with fits.open(file_path, memmap=True) as hdul:
            catalog_data = hdul

            if active_tab == 'tab_single_element':
                Emission_line = Emission_line_single
                if not Emission_line_single:
                    return dash.no_update
        
            elif active_tab == 'tab_combined_multi_element':
                Emission_line = Emission_line_multi
                if not Emission_line_multi:
                    return dash.no_update

            else:
                return dash.no_update
                
            waveMin_list = []
            waveMax_list = []
            fov_list = []
            sensitivity_list = []

            if band_data and band_data.get("waveMin"): #From the config sheet
                waveMin_list = band_data.get("waveMin", [])
                waveMax_list = band_data.get("waveMax", [])
                fov_list = band_data.get("fov", [])
                sensitivity_list = band_data.get("sensitivity", [])
            else: #From the manual input
                waveMin_list = [waveMin]
                waveMax_list = [waveMax]
                fov_list = [fov]
                sensitivity_list = [sensitivity]
                
            tobsv = tobsv * u.hr
            area = area * u.degree**2
            
            catIndices = []
            
            for w_min, w_max, FOV, sens in zip(waveMin_list, waveMax_list, fov_list, sensitivity_list):
                
                waveMin = float(w_min) * u.micron
                waveMax = float(w_max) * u.micron
                sensitivity = float(sens) * u.W / u.m**2
                fov = float(FOV) * u.degree**2
        
                band_indices = Detected_Galaxies(
                    Element=Emission_line,
                    data_table=catalog_data,
                    waveMin=waveMin,
                    waveMax=waveMax,
                    sensitivity=sensitivity,
                    tobsv=tobsv,
                    fov=fov,
                    area=area,
                    rest_wave_table=rest_wave_table,
                    upper_f_agn=up_f_agn,
                    lower_f_agn=low_f_agn,
                    joint=joint
                )

                catIndices.append(band_indices)
                
            return catIndices
        
    except Exception as error_data: 
        traceback.print_exc()
        print('Unable to retrive FITS data. Does your file path exist? Additionally, make sure data is stored in FITS[1]') 
        return None

In [11]:
@callback(
    Output(component_id='joint_store', component_property='data'),
    Input(component_id='detections_filter_2', component_property='value'), 
    prevent_initial_call=True)

def joint_filter(click):
    if ctx.triggered_id == "detections_filter_2" and click == "Joint": 
        return True 
        
    return False 
        
    

In [12]:
#Callback for pulling x_axis & y_axis headers from FITS
@callback(
    Output(component_id='dropdown_x', component_property='options'),
    Output(component_id='dropdown_x', component_property='value'),
    Output(component_id='dropdown_y', component_property='options'),
    Output(component_id='dropdown_y', component_property='value'),
    Input(component_id='catalog_data_store', component_property='data')) 

def dropdown_xy_single(file_path):
    
    if not file_path: 
        return [], None, [], None    
        
    try: 
        
        with fits.open(file_path, memmap=True) as hdul: 
            axes_column_headers = []
            for axes_headers in hdul:
                if hasattr(axes_headers, 'columns') and axes_headers.columns is not None:
                    axes_column_headers = axes_headers.columns.names
                    break
                    
        xy_options = []
        for xy in axes_column_headers:
            if not '[' in xy:
                xy_options.append(xy)
        
        if len(xy_options) == 0: 
            print("Error: Column headers must not include '[]', e.g., dec & NOT [dec]" ) 
            return [], None, [], None
            
        #Feed to dropdown
        return xy_options, xy_options[0], xy_options, xy_options[0]
        
    except Exception as error_headers:
        print('Error reading FITS headers') 
        return ['Error reading file headers'], None, ['Error reading file headers'], None

In [13]:
#Callback for pulling x_axis & y_axis headers from FITS
@callback(
    Output(component_id='multi_dropdown_x', component_property='options'),
    Output(component_id='multi_dropdown_x', component_property='value'),
    Output(component_id='multi_dropdown_y', component_property='options'),
    Output(component_id='multi_dropdown_y', component_property='value'),
    Input(component_id='catalog_data_store', component_property='data'),
    Input(component_id='graph_tabs', component_property='active_tab')) 

def dropdown_xy_multi(file_path, active_tab):
    
    if not file_path: 
        return [], None, [], None
        
    if active_tab == 'tab_single_element':
        return [], None, [], None
        
    try: 
        
        with fits.open(file_path, memmap=True) as hdul: 
            axes_column_headers = []
            for axes_headers in hdul:
                if hasattr(axes_headers, 'columns') and axes_headers.columns is not None:
                    axes_column_headers = axes_headers.columns.names
                    break
                    
        xy_options = []
        for xy in axes_column_headers:
            if not '[' in xy:
                xy_options.append(xy)
        
        if len(xy_options) == 0: 
            print("Error: Column headers must not include '[]', e.g., dec & NOT [dec]" ) 
            return [], None, [], None
            
        #Feed to dropdown
        return xy_options, xy_options[0], xy_options, xy_options[0]
        
    except Exception as error_headers:
        print('Error reading FITS headers') 
        return ['Error reading file headers'], None, ['Error reading file headers'], None

In [14]:
#Callback for reading configuration files
@callback(
    Output(component_id='configs_data', component_property='data'),
    Input(component_id='configs_selection', component_property='value'), 
    prevent_initial_call=False)

def dcc_configs_data(file_path):
    if not file_path:
        return no_update
        
    file_store = file_path
    
    return file_store

In [15]:
#Callback for Band selection single tab
@callback(
    Output(component_id='band_selection', component_property='options'), 
    Output(component_id='band_selection', component_property= 'value'), 
    Input(component_id='configs_data', component_property='data'),
    Input(component_id='graph_tabs', component_property='active_tab'),
    Input(component_id="clear_inputs", component_property="n_clicks"),)

 
def bands_single(configs_file, active_tab, clear_inputs): 

    if ctx.triggered_id == 'clear_inputs' and (clear_inputs is None or clear_inputs == 0):
        return dash.no_update, dash.no_update

    if ctx.triggered_id == 'clear_inputs': #clears bands and sets to None or the placeholder (change in layout)
        return dash.no_update, None

    
    #return empty lists when no file
    if not configs_file: 
        return [], None
    
    if active_tab == 'tab_combined_multi_element':
        return [], None
    try: 
        file_path = configs_file

        astropy_table = Table.read(file_path, format='ascii.ecsv')
        
        configs_read = astropy_table.to_pandas()
        band_names = ['band', 'BAND', 'bands', 'BANDS', 'Bands', 'Band'] #Allowed column names for band
        correct_band_names = None
    
        for name in band_names:
            if name in configs_read.columns:
                correct_band_names = name
                break
                
        else:
            if correct_band_names is None:
                raise KeyError("Error:No Bands found. Ensure the Band column is defined as 'band', 'BAND', 'bands', 'BANDS', 'Bands', or 'Band' in your csv.")


        extracted_bands = configs_read[correct_band_names].dropna().unique().astype(str).tolist()

        dropdown_options = [{'label': band, 'value': band} for band in extracted_bands]
        first_band = extracted_bands[0] 
        
        return dropdown_options, [first_band]
        
    except Exception as error_csv: 
        traceback.print_exc()
        print(f"Error reading in bands from ecsv: {error_csv}") 
        return ['Error reading in bands from ecsv'], []


In [16]:
#Callback for Band selection multi tab
@callback(
    Output(component_id='band_selection_2', component_property='options'), 
    Output(component_id='band_selection_2', component_property= 'value'), 
    Input(component_id='configs_data', component_property='data'),
    Input(component_id='graph_tabs', component_property='active_tab'),
    Input(component_id="clear_inputs_2", component_property="n_clicks"),
)

 
def bands_multi(configs_file, active_tab, clear_inputs):

    if ctx.triggered_id == 'clear_inputs_2' and (clear_inputs is None or clear_inputs == 0):
        return dash.no_update, dash.no_update

    if ctx.triggered_id == 'clear_inputs_2': #clears bands and sets to None or the placeholder (change in layout)
        return dash.no_update, None

    if not configs_file: 
        return [], None
    
    if active_tab == 'tab_single_element':
        return [], None
    try: 
        file_path = configs_file
        astropy_table = Table.read(file_path, format='ascii.ecsv')
        
        configs_read = astropy_table.to_pandas()
        band_names = ['band', 'BAND', 'bands', 'BANDS', 'Bands', 'Band'] #Allowed column names for band
        correct_band_names = None
    
        for name in band_names:
            if name in configs_read.columns:
                correct_band_names = name
                break
                
        else:
            if correct_band_names is None:
                raise KeyError("Error:No Bands found. Ensure the Band column is defined as 'band', 'BAND', 'bands', 'BANDS', 'Bands', or 'Band' in your csv.")


        extracted_bands = configs_read[correct_band_names].dropna().unique().astype(str).tolist()

        dropdown_options = [{'label': band, 'value': band} for band in extracted_bands]
        first_band = extracted_bands[0] 
        
        return dropdown_options, [first_band]
        
    except Exception as error_csv: 
        traceback.print_exc()
        print(f"Error reading in bands from ecsv: {error_csv}") 
        return ['Error reading in bands from ecsv'], []


In [17]:
#

@callback(
    Output(component_id='band_data', component_property='data'),
    Input(component_id='band_selection', component_property='value', allow_optional=True),
    Input(component_id='band_selection_2', component_property='value', allow_optional=True),
    Input(component_id='configs_selection', component_property='value'),
    Input(component_id='graph_tabs', component_property='active_tab'),
)
def band_input_update(band_single, band_multi, configs_file, active_tab):

    
    if not configs_file:
        return no_update

    if active_tab == 'tab_single_element':
        selected_band = band_single
        
    elif active_tab == 'tab_combined_multi_element':
        selected_band = band_multi
        
    else:
        return no_update

    if not selected_band:
        return no_update

    try:
        file_path = configs_file

        astropy_table = Table.read(file_path, format='ascii.ecsv')
        configs_read = astropy_table.to_pandas()

        band_names = ['band', 'BAND', 'bands', 'BANDS', 'Bands', 'Band'] 
        correct_band_names = None
    
        for name in band_names:
            if name in configs_read.columns:
                correct_band_names = name
                break
                

        if not isinstance(selected_band, list):
            selected_band = [selected_band]
            

        row_mask = configs_read[correct_band_names].astype(str).isin(selected_band)
        matched_rows = configs_read[row_mask]


        if matched_rows.empty:
            return no_update

        sens_names = ['sens', 'ps_sens']
        correct_sens_names = None
    
        for name in sens_names:
            if name in configs_read.columns:
                correct_sens_names = name
                break
                
        if correct_sens_names is None:
            return no_update

        waveMin_list = []
        waveMax_list = []
        fov_list = []
        sensitivity_list = []
        
        # from ECSV columns
        for index, row in matched_rows.iterrows():
            waveMin_list.append(float(row['low_wave']))
            waveMax_list.append(float(row['up_wave']))
            fov_list.append(float(row['FOV']))
            sensitivity_list.append(float(row[correct_sens_names]))

            

        band_extracted = {
            "waveMin": waveMin_list,
            "waveMax": waveMax_list,
            "fov": fov_list,
            "sensitivity": sensitivity_list}

        return band_extracted

    except Exception as error_csv: 
        traceback.print_exc()
        print(f"Error matching bands to ecsv variables: {error_csv}")
        return no_update


# Band

In [18]:
#graph multiple bands single tab
@callback(
    Output(component_id='active_band_toggle', component_property='options'),
    Output(component_id='active_band_toggle', component_property='value'),
    Input(component_id='band_selection', component_property='value', allow_optional=True),
    prevent_initial_call=True)

def toggle_options_single(selected_bands):
    
    if not selected_bands:
        return [], None
        
    if isinstance(selected_bands, list):
        bands_list = selected_bands
    else:
        bands_list = [selected_bands]
        
    toggle_options = []
    for bands in bands_list:
        band_info = {'label': bands, 'value': bands}
        toggle_options.append(band_info)
        
    if len(bands_list) > 0:
        default_selection = bands_list[0]
        
    else:
        default_selection = None
        
    return toggle_options, default_selection

In [19]:
#graph multiple bands multi tab
@callback(
    Output(component_id='active_band_toggle_2', component_property='options'),
    Output(component_id='active_band_toggle_2', component_property='value'),
    Input(component_id='band_selection_2', component_property='value', allow_optional=True),
    prevent_initial_call=True)

def toggle_options_multi(selected_bands):
    
    if not selected_bands:
        return [], None
        
    if isinstance(selected_bands, list):
        bands_list = selected_bands
    else:
        bands_list = [selected_bands]
        
    toggle_options = []
    for bands in bands_list:
        band_info = {'label': bands, 'value': bands}
        toggle_options.append(band_info)
        
    if len(bands_list) > 0:
        default_selection = bands_list[0]
        
    else:
        default_selection = None
        
    return toggle_options, default_selection

# Clear Single

In [20]:
@callback(
    Output(component_id='waveMin', component_property='value', allow_duplicate=True),
    Output(component_id='waveMax', component_property='value', allow_duplicate=True),
    Output(component_id='fov', component_property='value', allow_duplicate=True),
    Output(component_id='sensitivity', component_property='value', allow_duplicate=True),
    Input(component_id='clear_inputs', component_property='n_clicks'),
    prevent_initial_call=True)

def clear_bands (n_clicks):
    if n_clicks is None or n_clicks == 0:
        return dash.no_update

    return None, None, None, None
    

# Clear Multi

In [21]:
@callback(
    Output(component_id='waveMin', component_property='value', allow_duplicate=True),
    Output(component_id='waveMax', component_property='value', allow_duplicate=True),
    Output(component_id='fov', component_property='value', allow_duplicate=True),
    Output(component_id='sensitivity', component_property='value', allow_duplicate=True),
    Input(component_id='clear_inputs_2', component_property='n_clicks'),
    prevent_initial_call=True)

def clear_bands (n_clicks):
    if n_clicks is None or n_clicks == 0:
        return dash.no_update

    return None, None, None, None

# Callback to visually populate input boxes based on the selected Band(s) toggle
## Single Line Band Menu

In [22]:
#Callback to visually populate input boxes based on the selected Radio Item (Band) for single tab
@callback(
    Output(component_id='waveMin', component_property='value', allow_duplicate=True),
    Output(component_id='waveMax', component_property='value', allow_duplicate=True),
    Output(component_id='fov', component_property='value', allow_duplicate=True),
    Output(component_id='sensitivity', component_property='value', allow_duplicate=True),
    Input(component_id='active_band_toggle', component_property='value', allow_optional=True),
    Input(component_id='active_band_toggle', component_property='options', allow_optional=True),
    State(component_id='band_data', component_property='data'),
    prevent_initial_call=True)

def band_visual_single(active_band, all_bands, band_data):
    if not band_data or not band_data.get("waveMin") or not active_band or not all_bands:
        return no_update, no_update, no_update, no_update

    try:
        idx = 0
        for position, option in enumerate(all_bands):
            if option['value'] == active_band:
                idx = position
                break

        band_min = band_data["waveMin"][idx]
        band_max = band_data["waveMax"][idx]
        band_fov = band_data["fov"][idx]
        band_sens = band_data["sensitivity"][idx]

        return band_min, band_max, band_fov, band_sens

    except Exception:
        # If any index mismatch happens, safely change nothing
        return no_update, no_update, no_update, no_update


## Multi-Line Band Menu

In [23]:
#Callback to visually populate input boxes based on the selected Radio Item (Band) for multi tab
@callback(
    Output(component_id='waveMin', component_property='value', allow_duplicate=True),
    Output(component_id='waveMax', component_property='value', allow_duplicate=True),
    Output(component_id='fov', component_property='value', allow_duplicate=True),
    Output(component_id='sensitivity', component_property='value', allow_duplicate=True),
    Input(component_id='active_band_toggle_2', component_property='value', allow_optional=True),
    Input(component_id='active_band_toggle_2', component_property='options', allow_optional=True),
    State(component_id='band_data', component_property='data'),
    prevent_initial_call=True)

def band_visual_single(active_band, all_bands, band_data):
    if not band_data or not band_data.get("waveMin") or not active_band or not all_bands:
        return no_update, no_update, no_update, no_update

    try:
        idx = 0
        for position, option in enumerate(all_bands):
            if option['value'] == active_band:
                idx = position
                break

        band_min = band_data["waveMin"][idx]
        band_max = band_data["waveMax"][idx]
        band_fov = band_data["fov"][idx]
        band_sens = band_data["sensitivity"][idx]

        return band_min, band_max, band_fov, band_sens

    except Exception:
        # If any index mismatch happens, safely change nothing
        return no_update, no_update, no_update, no_update


# Callback Axes Data 
## retrieve axes data for single line emissions

In [24]:
#Callback to retrieve axes data for single line emissions
@callback(
    Output(component_id='GalaxyCat_data', component_property='data'), 
    Input(component_id='function_data', component_property='data'),
    Input(component_id='catalog_data_store', component_property='data'),
    Input(component_id='dropdown_x', component_property='value', allow_optional=True),
    Input(component_id='dropdown_y', component_property='value', allow_optional=True),
    Input(component_id='coloring', component_property='value', allow_optional=True))

def graph_data_catalog_single(detections, file_path, x_axes, y_axes, coloring):
    
    if not detections or not file_path or not x_axes or not y_axes:
        raise dash.exceptions.PreventUpdate

    with fits.open(file_path, memmap=True) as hdul:
        xy_data = hdul[1].data 
    
        if isinstance(detections, list) and len(detections) > 0 and isinstance(detections[0], list):
            x_data = []
            y_data = []
            
            for band_indices in detections:
                if coloring == "Band":
                    
                    x_data.append(xy_data[x_axes][band_indices].tolist())
                    y_data.append(xy_data[y_axes][band_indices].tolist())
                    
                else:
                    x_data.extend(xy_data[x_axes][band_indices].tolist())
                    y_data.extend(xy_data[y_axes][band_indices].tolist())
                    
       #Fallback if Band loop doesn't run
        else:
            x_data = xy_data[x_axes][detections].tolist()
            y_data = xy_data[y_axes][detections].tolist()
    
    return {'x_data': x_data, 'y_data': y_data}

## Callback to retrieve axes data for multi-line emissions

In [25]:

@callback(
    Output(component_id='GalaxyCat_data_2', component_property='data'), 
    Input(component_id='function_data', component_property='data'),
    Input(component_id='catalog_data_store', component_property='data'),
    Input(component_id='multi_dropdown_x', component_property='value', allow_optional=True),
    Input(component_id='multi_dropdown_y', component_property='value', allow_optional=True),
    Input(component_id='coloring_2', component_property='value', allow_optional=True), 
    Input(component_id='band_selection_2', component_property='value', allow_optional=True),
    Input(component_id='joint_store', component_property='data', allow_optional=True)
)


def graph_data_catalog_multi(detections, file_path, x_axes, y_axes, coloring, band_selection, joint):

    if not detections or not file_path or not x_axes or not y_axes:
        raise dash.exceptions.PreventUpdate

    def flatten_list(nested):
        if not isinstance(nested, list):
            yield nested
            return
        for item in nested:
            if isinstance(item, list):
                yield from flatten_list(item)
            else:
                yield item

    x_data = []
    y_data = []
    
    # Check if data exist as a list or list of lists
    if isinstance(x_axes, list) and len(x_axes) > 0:
        clean_x = x_axes[0]
    else:
        clean_x = x_axes

    if isinstance(y_axes, list) and len(y_axes) > 0:
        clean_y = y_axes[0]
    else:
        clean_y = y_axes

    with fits.open(file_path, memmap=True) as hdul:
        xy_data = hdul[1].data

    if joint:
        flat_detections = list(flatten_list(detections))

        iterable_flat = flat_detections if isinstance(flat_detections, list) else [flat_detections]
        row_indices = [int(xy) for xy in iterable_flat if xy is not None]
        
        if coloring == "Band":
            x_data = []
            y_data = []
            bands_to_use = band_selection if band_selection else [0]
            for band in bands_to_use:
                if len(row_indices) > 0:
                    x_data.append(xy_data[clean_x][row_indices].tolist())
                    y_data.append(xy_data[clean_y][row_indices].tolist())
                else:
                    x_data.append([])
                    y_data.append([])
                    
        elif coloring == "None" or coloring == "Line":
            temp_x = []
            temp_y = []
            if len(row_indices) > 0:
                temp_x.extend(xy_data[clean_x][row_indices].tolist())
                temp_y.extend(xy_data[clean_y][row_indices].tolist())
            x_data = [temp_x]
            y_data = [temp_y]

    elif isinstance(detections, list) and len(detections) > 0 and isinstance(detections[0], list) and len(detections[0]) > 0 and isinstance(detections[0][0], list):
        num_emission_lines = len(detections[0])

        if coloring == "Line":
            x_data = [[] for _ in range(num_emission_lines)]
            y_data = [[] for _ in range(num_emission_lines)]
        
            for bands in detections:
                if isinstance(bands, list):
                    for line_idx, lines in enumerate(bands):
                        if lines and line_idx < num_emission_lines:
                            iterable_lines = lines if isinstance(lines, list) else [lines]
                            row_indices = [int(xy) for xy in iterable_lines if xy is not None]
                            if len(row_indices) > 0:
                                x_data[line_idx].extend(xy_data[clean_x][row_indices].tolist())
                                y_data[line_idx].extend(xy_data[clean_y][row_indices].tolist())
                
        elif coloring == "Band":
            x_data = []
            y_data = []
    
            for bands in detections:
                band_x = []
                band_y = []
                if isinstance(bands, list):
                    for lines in bands:
                        if lines:

                            iterable_lines = lines if isinstance(lines, list) else [lines]
                            row_indices = [int(xy) for xy in iterable_lines if xy is not None]
                            if len(row_indices) > 0:
                                band_x.extend(xy_data[clean_x][row_indices].tolist())
                                band_y.extend(xy_data[clean_y][row_indices].tolist())
                
                x_data.append(band_x)
                y_data.append(band_y)

        elif coloring == "None":
            x_data = [[]] 
            y_data = [[]]
    
            for bands in detections:
                if isinstance(bands, list):
                    for lines in bands:
                        if lines:

                            iterable_lines = lines if isinstance(lines, list) else [lines]
                            row_indices = [int(xy) for xy in iterable_lines if xy is not None]
                            if len(row_indices) > 0:
                                x_data[0].extend(xy_data[clean_x][row_indices].tolist())
                                y_data[0].extend(xy_data[clean_y][row_indices].tolist())

        else:
            x_data = []
            y_data = []
            for bands in detections:
                band_x = []
                band_y = []
                if isinstance(bands, list):
                    for lines in bands:
                        if lines:

                            iterable_lines = lines if isinstance(lines, list) else [lines]
                            row_indices = [int(xy) for xy in iterable_lines if xy is not None]
                            if len(row_indices) > 0:
                                band_x.extend(xy_data[clean_x][row_indices].tolist())
                                band_y.extend(xy_data[clean_y][row_indices].tolist())
                x_data.append(band_x)
                y_data.append(band_y)

    else:
        x_data = []
        y_data = []
        for bands in detections:
            band_x = []
            band_y = []
            if isinstance(bands, list):
                for lines in bands:
                    if lines:

                        iterable_lines = lines if isinstance(lines, list) else [lines]
                        row_indices = [int(xy) for xy in iterable_lines if xy is not None]
                        if len(row_indices) > 0:
                            band_x.extend(xy_data[clean_x][row_indices].tolist())
                            band_y.extend(xy_data[clean_y][row_indices].tolist())
            x_data.append(band_x)
            y_data.append(band_y)

    return {'x_data': x_data, 'y_data': y_data}

# Graphing Callbacks
## Single Line Graphs

In [26]:
#Callback for single line emission graphs
@callback(
    Output(component_id='controls-and-graph', component_property='figure'),
    Input(component_id='graph_button', component_property='n_clicks'), 
    State(component_id='GalaxyCat_data', component_property='data'), 
    State(component_id='graph_type', component_property='value'),
    State(component_id='dropdown_x', component_property='value', allow_optional=True), 
    State(component_id='dropdown_y', component_property='value', allow_optional=True),
    State(component_id='axis_type', component_property='value', allow_optional=True),
    State(component_id='controls-and-radio-item', component_property='value', allow_optional=True),
    State(component_id='coloring', component_property='value', allow_optional=True),
    State(component_id='band_selection', component_property='value', allow_optional=True),
    State(component_id='Hist_Tools', component_property='value', allow_optional=True), 
    State(component_id='catalog_data_store', component_property='data'),
)
def update_graph(n_clicks, axis, graph_type, x_axes, y_axes, axis_type, element_name, coloring, band_selection, hist_tools, file_path):
    #So graph doesn't automatically get graphed & None value protection
    if not n_clicks or ctx.triggered_id != 'graph_button':
        raise dash.exceptions.PreventUpdate
        
    # None value & data fallback
    if not axis or not axis_type or 'x_data' not in axis or 'y_data' not in axis:
        raise dash.exceptions.PreventUpdate

    # Define title strings
    clean_title_x = x_axes if isinstance(x_axes, str) else 'X Axis'
    clean_title_y = y_axes if isinstance(y_axes, str) else 'Y Axis'

    #Scaling 
    graph_scalex = axis_type in ['All Log', 'X-Log'] #True False logic, adding extra scaleing options possible
    graph_scaley = axis_type in ['All Log', 'Y-Log']

    fig = go.Figure()

    annotation = []

    if coloring == 'Band':
        x_components = axis.get('x_data', [])
        y_components = axis.get('y_data', [])
        
        # Ensure band_selection is an iterable list
        bands = band_selection if isinstance(band_selection, list) else [band_selection]

        all_combined_data = []
        
        for x in x_components:
            if x is not None and len(x) > 0:
                if graph_scalex:
                    all_combined_data.extend(np.log10(x).tolist())
                else:
                    all_combined_data.extend(x)
                    
        if all_combined_data:
            global_min = np.min(all_combined_data)
            global_max = np.max(all_combined_data)
            global_bins = np.linspace(global_min, global_max, 101)
            global_bin_size = global_bins[1] - global_bins[0]
        else:
            global_bins = 100
            global_bin_size = None

        # Conditon for Histogram
        All_Sources = True
        all_bandsloop_min = []
        all_bandsloop_max = []
        
        colors = px.colors.qualitative.Set1
        
        
        # Band Loop
        for i, (band, x_data, y_data) in enumerate(zip(bands, x_components, y_components)):
            if x_data is None or len(x_data) == 0:
                continue
            
            #Trace naming per band
            if isinstance(element_name, list):
                trace_name = element_name[i] if i < len(element_name) else str(band)
            else:
                trace_name = f"{element_name} ({band})" if element_name else str(band)

            # Histogram
            
            if graph_type == 'Histogram':

                arr_data = np.asarray(x_data, dtype=float)
                valid_mask = np.isfinite(arr_data) & (arr_data > 0)
                filtered_data = arr_data[valid_mask]

                if len(filtered_data) == 0:
                    continue

                x_hist = np.log10(filtered_data).tolist() if graph_scalex else filtered_data.tolist()

                all_bandsloop_min.append(np.min(x_hist))
                all_bandsloop_max.append(np.max(x_hist))
                
                if graph_scalex:
                    annotation.append(dict(x=1.03, y=0, 
                    xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                    showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))

                num, bins = np.histogram(x_hist, bins=global_bins)
                bin_centers = 0.5 * (bins[:-1] + bins[1:])
                
                fig.add_trace(go.Histogram(x=x_hist, xbins=dict(start=global_bins[0], end=global_bins[-1], size=global_bin_size), 
                name=trace_name, bingroup=band, opacity=0.6, showlegend=True, marker_color=colors[i % len(colors)]))

                if "Survey Curve" in hist_tools:
            
                    fig.add_trace(go.Scatter(
                    x=bin_centers, 
                    y=num, 
                    mode='lines', 
                    name=trace_name,
                    line=dict(shape='linear', width=2.0), 
                    showlegend=True))

                if "All Sources" in hist_tools:
                    if not file_path:
                        raise dash.exceptions.PreventUpdate

                    with fits.open(file_path, memmap=True) as hdul:
                        xy_data = hdul[1].data
                        x_data_all = xy_data[x_axes]

                        arr_data_all = np.asarray(x_data_all, dtype=float)
                        valid_mask_all = np.isfinite(arr_data_all) & (arr_data_all > 0)
                        filtered_data_all = arr_data_all[valid_mask_all]
                        #cut_data_mask = (filtered_data_all >= np.min(filtered_data)) & (filtered_data_all <= np.max(filtered_data))
                        #cut_data_all = filtered_data_all[cut_data_mask]

                    if len(filtered_data_all) > 0:
                    
                        x_hist_all = np.log10(filtered_data_all) if graph_scalex else filtered_data_all

                        num_all, bins_all = np.histogram(x_hist_all, bins=global_bins)
                        bin_centers_all = 0.5 * (bins_all[:-1] + bins_all[1:])

                        fig.add_trace(go.Scatter(
                        x=bin_centers_all, y=num_all, mode='lines', 
                        name="All Sources",
                        line=dict(shape='hvh', width=2.0, color="Black"), 
                        showlegend=All_Sources, legendrank=1))

                        All_Sources = False

            # XY Scatter Plot
            elif graph_type == 'XY-Graph':
                fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name=trace_name, showlegend=True, marker_color=colors[i % len(colors)]))
            
            # 2D Histogram
            elif graph_type == '2D-Histogram':

                arr_data = np.asarray(x_data, dtype=float)
                arr_data_y = np.asarray(y_data, dtype=float)
                valid_mask = np.isfinite(arr_data) & (arr_data > 0)
                valid_mask_y = np.isfinite(arr_data_y) & (arr_data_y > 0)
                filtered_data = arr_data[valid_mask]
                filtered_data_y = arr_data_y[valid_mask_y]
                
                x_2d = np.log10(filtered_data).tolist() if graph_scalex else filtered_data.tolist()

                if graph_scalex:
                    annotation.append(dict(x=1.03, y=0, 
                        xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                        showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))  
                        
                y_2d = np.log10(filtered_data_y).tolist() if graph_scaley else filtered_data_y.tolist()

                
                if graph_scaley:
                    annotation.append(dict(x=-0.05, y=1.03, 
                        xref="paper", yref="paper", text="(Scale of 10<sup>y</sup>)", 
                        showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
        
                
                COLORSCALES = ["Aggrnyl", "Agsunset", "Blugrn", "Burg", "Burgyl", "Cividis", "Darkmint", "Emrld", "Inferno", "Magma", "Magenta", "Mint", "Oryel", "Peach", "Pinkyl", "Plasma", "Purpor", "Redor", "Sunset", "Sunsetdark", "Teal", "Tealgrn", "Viridis", "algae", "amp", "deep", "dense", "haline", "matter", "solar", "speed", "tempo"]
 
                chosen_color = COLORSCALES[i % len(COLORSCALES)] #use % to adjust loop to n of inputs
                
                fig.add_trace(go.Histogram2dContour(x=x_2d, y=y_2d, name=trace_name, nbinsx=100, nbinsy=100, autocontour=False, contours=dict(start=0, end=12, size=1.5, coloring='lines'), line=dict(width=2), showscale=False, showlegend=True, colorscale=chosen_color))

        if graph_type == 'Histogram':
            view_min = np.min(all_bandsloop_min)
            view_max = np.max(all_bandsloop_max)
            fig.update_layout(title=dict(text=f'Histogram of {clean_title_x}', font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title='Total Detections', yaxis_type='log' if graph_scaley else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black'), range=[view_min, view_max]),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')), barmode='overlay')

        elif graph_type == 'XY-Graph':
            fig.update_layout(title=dict(text=f'{clean_title_y} vs {clean_title_x}', font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title=clean_title_y, xaxis_type='log' if graph_scalex else 'linear', yaxis_type='log' if graph_scaley else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))

        elif graph_type == '2D-Histogram':
            fig.update_layout(margin=dict(r=200), title=dict(text=f'2D Histogram of {clean_title_x}', font=dict(color="black")), 
                annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title=clean_title_y, xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))

        return fig

    else:
        
        #Histogram single line "No coloring"
        if graph_type=='Histogram':

            arr_data = np.asarray(axis['x_data'], dtype=float)
            valid_mask = np.isfinite(arr_data) & (arr_data > 0)
            filtered_data = arr_data[valid_mask]

            if len(filtered_data) == 0:
                raise dash.exceptions.PreventUpdate

            if axis_type == 'All Log' or axis_type == 'X-Log':
   
                x_hist = np.log10(filtered_data).tolist()
                
                annotation.append(dict(x=1.03, y=0, 
                xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
                
            else:
                x_hist = filtered_data.tolist()


            num, bins = np.histogram(x_hist, bins=100)
            bin_centers = 0.5 * (bins[:-1] + bins[1:])
                
            fig.add_trace(go.Histogram(x=x_hist, xbins=dict(start=bins[0], end=bins[-1], size=(bins[1] - bins[0])), 
            name=element_name[0] if isinstance(element_name, list) else element_name, opacity=0.6, showlegend=True))

            if "Survey Curve" in hist_tools:
                
                fig.add_trace(go.Scatter(
                x=bin_centers, 
                y=num, mode='lines', 
                name=element_name[0] if isinstance(element_name, list) else element_name,
                line=dict(shape='linear', width=2.0), 
                showlegend=True))

            if "All Sources" in hist_tools:
                if not file_path:
                    raise dash.exceptions.PreventUpdate

                with fits.open(file_path, memmap=True) as hdul:
                    xy_data = hdul[1].data
                    x_data_all = xy_data[x_axes]


                    arr_data_all = np.asarray(x_data_all, dtype=float)
                    valid_mask_all = np.isfinite(arr_data_all) & (arr_data_all > 0)
                    filtered_data_all = arr_data_all[valid_mask_all]

                if len(filtered_data_all) > 0:

                    x_hist_all = np.log10(filtered_data_all).tolist() if graph_scalex else filtered_data_all.tolist()

                    num_all, bins_all = np.histogram(x_hist_all, bins=100)
                    bin_centers_all = 0.5 * (bins_all[:-1] + bins_all[1:])
                        
                    fig.add_trace(go.Scatter(
                    x=bin_centers_all, y=num_all, mode='lines', 
                    name= "All Sources",
                    line=dict(shape='hvh', width=2.0, color="Black"), 
                    showlegend=True, legendrank=1))
                    
            view_min = np.min(x_hist)
            view_max = np.max(x_hist)
            fig.update_layout(title=dict(text=f'Histogram of {x_axes}', font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=x_axes, yaxis_title='Total Detections', yaxis_type='log' if (axis_type =='All Log' or axis_type =='Y-Log') else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black'), range=[view_min, view_max]),
                    yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')), barmode='overlay')
            return fig        
        
        #For xy graph single line 
        elif graph_type=='XY-Graph':

            fig.add_trace(go.Scatter(x=axis['x_data'], y=axis['y_data'], mode='markers', name = element_name[0] if isinstance(element_name, list) else element_name, showlegend=True))
                
            fig.update_layout(title=dict(text=f'{y_axes} vs {x_axes}', font=dict(color="black")), legend=dict(font=dict(color="black")), xaxis_title = x_axes, yaxis_title = y_axes, xaxis_type='log' if graph_scalex else 'linear', yaxis_type='log' if graph_scaley else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))
                    
            return fig
        # For 2D histogram single line
        elif graph_type=='2D-Histogram':

            if axis_type == 'All Log' or axis_type == 'X-Log':

                arr_data = np.asarray(axis['x_data'], dtype=float)
                valid_mask = np.isfinite(arr_data) & (arr_data > 0)
                filtered_data = arr_data[valid_mask]

                if len(filtered_data) == 0:
                    raise dash.exceptions.PreventUpdate

                x_2dsingle = np.log10(filtered_data).tolist()
                
                annotation.append(dict(x=1.01, y=-0.05, 
                xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
            else:
                x_2dsingle = axis['x_data']
            if axis_type == 'All Log' or axis_type == 'Y-Log':

                arr_data = np.asarray(axis['x_data'], dtype=float)
                valid_mask = np.isfinite(arr_data) & (arr_data > 0)
                filtered_data = arr_data[valid_mask]

                if len(filtered_data) == 0:
                    raise dash.exceptions.PreventUpdate

                y_2dsingle = np.log10(filtered_data).tolist()

                annotation.append(dict(x=-0.05, y=1.03, 
                xref="paper", yref="paper", text="(Scale of 10<sup>y</sup>)", 
                showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
            else:
                y_2dsingle = axis['y_data']
            fig.add_trace(go.Histogram2dContour(x=x_2dsingle, y=y_2dsingle, nbinsx=100, nbinsy = 100, name=element_name[0] if isinstance(element_name, list) else element_name, autocontour=False, contours=dict(start=0, end=12, size=1.5, coloring='lines'), line=dict(width=2), showscale=False, showlegend=True))
            fig.update_layout(margin=dict(r=200), title=dict(text=f"2D Histogram of {x_axes}", font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=x_axes, yaxis_title=y_axes, xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))
            return fig
        

## Multi-line Graphs

In [27]:
#Graphs multi line emission
@callback(
    Output(component_id='controls-and-graph_2', component_property='figure'),
    Input(component_id='graph_button_2', component_property='n_clicks'), 
    State(component_id='GalaxyCat_data_2', component_property='data'), 
    State(component_id='graph_type', component_property='value'),
    State(component_id='multi_dropdown_x', component_property='value', allow_optional=True), 
    State(component_id='multi_dropdown_y', component_property='value',allow_optional=True),
    State(component_id='axis_type', component_property='value', allow_optional=True),
    State(component_id='multi-controls-and-radio-item', component_property='value', allow_optional=True),
    State(component_id='coloring_2', component_property='value', allow_optional=True),
    State(component_id='band_selection_2', component_property='value', allow_optional=True),
    State(component_id='joint_store', component_property='data', allow_optional=True), 
    State(component_id='Hist_Tools_2', component_property='value', allow_optional=True), 
    State(component_id='catalog_data_store', component_property='data')
    )

def update_graph_2(n_clicks, axis, graph_type, x_axes, y_axes, axis_type, element_name, coloring, band_selection, Joint, hist_tools, file_path):
    # So graph doesn't automatically get graphed & None value protection
    if n_clicks is None or n_clicks == 0:
        raise dash.exceptions.PreventUpdate
        
    if axis is None or axis_type is None or 'x_data' not in axis or 'y_data' not in axis:
        raise dash.exceptions.PreventUpdate
        
    # Scaling Titles
    if isinstance(x_axes, list) and len(x_axes) > 0:
        clean_title_x = x_axes[0]
    else:
        clean_title_x = x_axes

    if isinstance(y_axes, list) and len(y_axes) > 0:
        clean_title_y = y_axes[0]
    else:
        clean_title_y = y_axes
        
    # Scaling Flags
    graph_scalex = axis_type in ['All Log', 'X-Log']
    graph_scaley = axis_type in ['All Log', 'Y-Log']

    # So graph only gets updated when button is clicked 
    if 'graph_button_2' == ctx.triggered_id:

        fig = go.Figure()

        annotation = []

        x_components = axis['x_data']
        y_components = axis['y_data']

        print("All X component lengths:", [len(x) if x is not None else 'None' for x in x_components])

        if isinstance(element_name, str):
            element_name = [element_name]
        if isinstance(band_selection, str):
            band_selection = [band_selection]

        if coloring == "Band":
            labels = [band_selection[i] if i < len(band_selection) else f"Band {i+1}"
                              for i in range(len(x_components))]
                      
        elif coloring == "None":
            labels = ["All Detections"]

        elif Joint:
            labels = [", ".join(element_name)]

        else:
            labels = [element_name[i] if i < len(element_name) else f"Line {i+1}" 
                              for i in range(len(x_components))]

        all_combined_data = []
    
        for x_multi in x_components:
            if x_multi:
                if axis_type in ['All Log', 'X-Log']:
                    all_combined_data.extend(np.log10(x_multi).tolist())
                else:
                    all_combined_data.extend(x_multi)
                
        # global bin array
        if all_combined_data:
            global_min = np.min(all_combined_data)
            global_max = np.max(all_combined_data)
        
        # 100 bins (always desired bins +1)
            global_bins = np.linspace(global_min, global_max, 101)
            global_bin_size = global_bins[1] - global_bins[0]
        else:
            global_bins = 100
            global_bin_size = None

        All_Sources = True 
        all_loops_min = []
        all_loops_max = []

        colors = px.colors.qualitative.Set1
        
        
        for i, (elem, x_multi, y_multi) in enumerate(zip_longest(labels, x_components, y_components, fillvalue=None)):
            print(f"Loop Item -> Name: {elem}, X-Length: {len(x_multi) if x_multi is not None else 'None'}")
            
            if not x_multi or not y_multi:
                continue

            arr_data = np.asarray(x_multi, dtype=float)
            valid_mask = np.isfinite(arr_data) & (arr_data > 0)
            filtered_data = arr_data[valid_mask]

            arr_data_y = np.asarray(y_multi, dtype=float)
            valid_mask_y = np.isfinite(arr_data_y) & (arr_data_y > 0)
            filtered_data_y = arr_data_y[valid_mask_y]


            # For Histogram multi
            if graph_type == 'Histogram':
                
                
                if axis_type == 'All Log' or axis_type == 'X-Log':
                    x_hist = np.log10(filtered_data).tolist()
                    
                    annotation.append(dict(x=1.03, y=0, 
                    xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                    showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
                    
                else:
                    x_hist = filtered_data.tolist()

                all_loops_min.append(np.min(x_hist))
                all_loops_max.append(np.max(x_hist))

                num, bins = np.histogram(x_hist, bins=global_bins)
                bin_centers = 0.5 * (bins[:-1] + bins[1:])
                bin_size = bins[1] - bins[0]
                
                fig.add_trace(go.Histogram(x=x_hist, xbins=dict(start=bins[0], end=bins[-1], size=bin_size), 
                                           marker_color=colors[i % len(colors)], marker_line_color=colors[i % len(colors)],
                                           name=elem, bingroup=elem, opacity=0.6, showlegend=True))

                if "Survey Curve" in hist_tools:
            
                    fig.add_trace(go.Scatter(
                    x=bin_centers, 
                    y=num, 
                    mode='lines', 
                    name=elem,
                    line=dict(shape='linear', width=2.0), 
                    showlegend=True))

                if "All Sources" in hist_tools:
                    if not file_path:
                        raise dash.exceptions.PreventUpdate

                    with fits.open(file_path, memmap=True) as hdul:
                        xy_data = hdul[1].data
                        x_data_all = xy_data[x_axes]

                        arr_data_all = np.asarray(x_data_all, dtype=float)
                        valid_mask_all = np.isfinite(arr_data_all) & (arr_data_all > 0)
                        filtered_data_all = arr_data_all[valid_mask_all]

                    if len(filtered_data_all) > 0:

                        x_hist_all = np.log10(filtered_data_all).tolist() if graph_scalex else filtered_data_all.tolist()

                        num_all, bins_all = np.histogram(x_hist_all, bins=global_bins)
                        bin_centers_all = 0.5 * (bins_all[:-1] + bins_all[1:])
                        
                        fig.add_trace(go.Scatter(
                        x=bin_centers_all, y=num_all, mode='lines', 
                        name= "All Sources",
                        line=dict(shape='hvh', width=2.0, color="Black"), 
                        showlegend=All_Sources, legendrank=1))
                        
                        All_Sources = False
        
            # For xy graph multi lines 
            elif graph_type == 'XY-Graph':
                fig.add_trace(go.Scatter(x=x_multi, y=y_multi, mode='markers', name=elem, showlegend=True, 
                                        marker_color=colors[i % len(colors)]))
                    
            # For 2D histogram multi lines contour map
            elif graph_type == '2D-Histogram':
                if axis_type == 'All Log' or axis_type == 'X-Log':

                    
                    x_2dsingle = np.log10(filtered_data).tolist()

                    annotation.append(dict(x=1.01, y=-0.05, 
                    xref="paper", yref="paper", text="(Scale of 10<sup>x</sup>)", 
                    showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
                else:
                    x_2dsingle = x_multi
                if axis_type == 'All Log' or axis_type == 'Y-Log':

                    y_2dsingle = np.log10(filtered_data_y).tolist()

                    annotation.append(dict(x=-0.05, y=1.03, 
                    xref="paper", yref="paper", text="(Scale of 10<sup>y</sup>)", 
                    showarrow=False, xanchor="left", yanchor="middle", font=dict(color="black")))
                else:
                    y_2dsingle = y_multi
                fig.add_trace(go.Histogram2dContour(x=x_2dsingle, y=y_2dsingle, nbinsx=100, nbinsy=100, name=elem, autocontour=False, contours=dict(start=0, end=12, size=1.5, coloring='lines'), line=dict(width=2), showscale=False, showlegend=True))

        if graph_type == 'Histogram':
            view_min = np.min(all_loops_min)
            view_max = np.max(all_loops_max)
            fig.update_layout(title=dict(text=f'Histogram of {clean_title_x}', font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title='Total Detections', yaxis_type='log' if (axis_type == 'All Log' or axis_type == 'Y-Log') else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black'), range=[view_min, view_max]),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')), barmode='overlay')
            return fig
            
        elif graph_type == 'XY-Graph':
            fig.update_layout(title=dict(text=f'{clean_title_y} vs {clean_title_x}', font=dict(color="black")), legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title=clean_title_y, xaxis_type='log' if graph_scalex else 'linear', yaxis_type='log' if graph_scaley else 'linear', xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))
            return fig
            
        elif graph_type == '2D-Histogram':
            fig.update_layout(margin=dict(r=200), title=dict(text=f'2D Histogram of {clean_title_x}', font=dict(color="black")), annotations = annotation, legend=dict(font=dict(color="black")), xaxis_title=clean_title_x, yaxis_title=clean_title_y, xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
                yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')))
            return fig

    # Fallback
    raise dash.exceptions.PreventUpdate


# FITS Download
## Single Line Download

In [28]:
#Callback to download detections as FITS for single line emissions
@callback(
    Output(component_id='Filtered_galaxies', component_property='data'), 
    Input(component_id='FITS_download', component_property='n_clicks'),
    Input(component_id='catalog_data_store', component_property='data'),
    State(component_id='function_data', component_property='data'),
    prevent_initial_call=True)
                 
          
def FITS_Download(clicks_button, file_path, detections):
    
    if not clicks_button or not detections:
        #print('Error saving detections. Check if outputs are returned properly by "function_data"')
        return no_update
        
    if ctx.triggered_id == 'FITS_download':
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
        file_name = f"Filtered_Detections_{timestamp}.fits"
        local_file_path = os.path.join(DOWNLOAD_FOLDER, file_name)

        try:
            with fits.open(file_path, memmap=True) as hdul:
                fits_data = hdul[1].data
                original_header = hdul[1].header

                if isinstance(detections, list) and len(detections) > 0 and isinstance(detections[0], list):
                    stripped_list = detections[0]
                else:
                    stripped_list = detections
                
                download_data = fits_data[stripped_list]
                table = Table(download_data)
                
        except Exception as reading_error:
            print('Error slicing data from catalog. Table conversion has failed.')
            raise dash.exceptions.PreventUpdate

        try:
            hdu_0 = fits.PrimaryHDU()
            hdu_1 = fits.BinTableHDU(data=table, header=original_header)
            hdul_new = fits.HDUList([hdu_0, hdu_1])
        
            hdul_new.writeto(local_file_path)
            print(f"FITS Download successful: {local_file_path}")
            return dash.dcc.send_file(local_file_path)
            
        except Exception as file_error:
            print('Error exporting FITS file')
            raise dash.exceptions.PreventUpdate
            
    return no_update

    

## Multi-Line Download

In [29]:
#Callback to download detections as FITS for multi line emissions
@callback(
    Output(component_id='Filtered_galaxies_2', component_property='data'), 
    Input(component_id='FITS_download_2', component_property='n_clicks', allow_optional=True),
    Input(component_id='catalog_data_store', component_property='data'),
    State(component_id='function_data', component_property='data'),
    prevent_initial_call=True)
                 
          
def FITS_Download(clicks_button, file_path, detections):
    
    if not clicks_button or not detections:
        #print('Error saving detections. Check if outputs are returned properly by "function_data"')
        return no_update
        
    if ctx.triggered_id == 'FITS_download_2':
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H.%M.%S") #change to paramter inputs (what was graphed)
        file_name = f"Filtered_Detections_{timestamp}.fits"
        local_file_path = os.path.join(DOWNLOAD_FOLDER, file_name)

        try:
            with fits.open(file_path, memmap=True) as hdul:
                fits_data = hdul[1].data
                original_header = hdul[1].header

                galaxy_filter = []
                for lists in detections:
                    for rows in lists:
                        galaxy_filter.append(rows)

                stripped_list = list(set(galaxy_filter)) #set is important to eliminate duplicate indices 
                download_data = fits_data[stripped_list]
                table = Table(download_data)
                
        except Exception as reading_error:
            print('Error slicing data from catalog. Table conversion has failed.')
            raise dash.exceptions.PreventUpdate

        try:
            hdu_0 = fits.PrimaryHDU()
            hdu_1 = fits.BinTableHDU(data=table, header=original_header)
            hdul_new = fits.HDUList([hdu_0, hdu_1])
        
            hdul_new.writeto(local_file_path)   #add overwrite=True to get rid of duplicate download in folder
            print(f"FITS Download successful: {local_file_path}")
            return dash.dcc.send_file(local_file_path)
            
        except Exception as file_error:
            print('Error exporting FITS file')
            raise dash.exceptions.PreventUpdate
            
    return no_update

# Run The App/Server

In [30]:
# Run the app
if __name__ == '__main__':
    app.run(debug=False, port=2225, use_reloader=False, dev_tools_ui=True, dev_tools_props_check=True)

Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [NeIII] 16
Processing element: [SIII] 19
Processing element: [NeIII] 16
All X component lengths: [1977]
Loop Item -> Name: [SIII] 19, [NeIII] 16, X-Length: 1977
All X component lengths: [1977, 1977]
Loop Item -> Name: Band 1, X-Length: 1977
Loop Item -> Name: Band 2, X-Length: 1977
Processing element: [SIII] 19
Process

/home/wag_user2/miniforge3/envs/astropy/lib/python3.13/site-packages/astropy/units/quantity.py:659: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19


/home/wag_user2/miniforge3/envs/astropy/lib/python3.13/site-packages/astropy/units/quantity.py:659: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [NeIII] 16
Processing element: [SIII] 19
Processing element: [NeIII] 16
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
Processing element: [SIII] 19
